In [1]:
import pandas as pd

In [2]:
c = pd.read_csv("E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\customers.csv")

In [3]:
c.columns

Index(['customer_id', 'name', 'city', 'signup_date', 'email'], dtype='str')

In [4]:
c.head()

,customer_id,name,city,signup_date,email
0,CUST001,Rohan Iyer,Kolkata,2023-04-16,user1@example.com
1,CUST002,Aditya Shah,Mumbai,2022-06-11,user2@example.com
2,CUST003,Arjun Singh,Pune,2023-03-15,user3@example.com
3,CUST004,Neha Joshi,Mumbai,2022-11-06,user4@example.com
4,CUST005,Divya Joshi,Mumbai,2023-11-02,user5@example.com


In [5]:
p = pd.read_parquet("E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\products.parquet", engine='fastparquet')

In [6]:
p.columns

Index(['line_item_id', 'order_id', 'product_id', 'product_name', 'category',
       'quantity', 'unit_price', 'total_price'],
      dtype='str')

In [7]:
p.head()

,line_item_id,order_id,product_id,product_name,category,quantity,unit_price,total_price
0,LI0001,ORD2007,PR02,Cotton Kurti,Clothing,4,899,3596
1,LI0002,ORD2057,PR07,Yoga Mat,Sports,1,699,699
2,LI0003,ORD2071,PR05,Python Book,Books,4,799,3196
3,LI0004,ORD2090,PR03,Basmati Rice 5kg,Grocery,2,450,900
4,LI0005,ORD2089,PR01,Wireless Earbuds,Electronics,3,2499,7497


In [8]:
j = pd.read_json('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\orders.json')

In [9]:
j.columns

Index(['order_id', 'customer_id', 'order_date', 'status', 'total_amount',
       'num_items'],
      dtype='str')

In [10]:
j.head()

,order_id,customer_id,order_date,status,total_amount,num_items
0,ORD2001,CUST028,2023-05-20,delivered,2397,1
1,ORD2002,CUST032,2023-08-22,delivered,6197,3
2,ORD2003,CUST037,2023-12-12,delivered,9897,1
3,ORD2004,CUST003,2023-01-07,shipped,5596,2
4,ORD2005,CUST012,2023-01-19,processing,15792,4


In [11]:
import duckdb

con = duckdb.connect()

# Q1
q1 = """
SELECT 
    c.customer_id,
    c.name,
    COUNT(j.order_id) AS total_orders
FROM read_csv_auto('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\customers.csv') c
LEFT JOIN read_json_auto('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\orders.json') j
ON c.customer_id = j.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_orders DESC;
"""
df1 = con.execute(q1).fetchdf()
df1

,customer_id,name,total_orders
0,CUST048,Suresh Menon,6
1,CUST004,Neha Joshi,6
2,CUST011,Rohan Mehta,5
3,CUST016,Vikram Singh,4
4,CUST032,Kiran Chopra,4
5,CUST025,Aarav Desai,4
6,CUST006,Neha Sharma,4
7,CUST012,Sneha Patel,4
8,CUST017,Nikhil Mehta,3
9,CUST041,Lakshmi Kumar,3


In [12]:
q2 = """
SELECT 
    c.customer_id,
    c.name,
    SUM(j.total_amount) AS total_spent
FROM read_csv_auto('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\customers.csv') c
JOIN read_json_auto('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\orders.json') j
ON c.customer_id = j.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_spent DESC
LIMIT 3;
"""
df2 = con.execute(q2).fetchdf()
df2

,customer_id,name,total_spent
0,CUST025,Aarav Desai,50331.0
1,CUST004,Neha Joshi,45527.0
2,CUST048,Suresh Menon,40629.0


In [13]:
q3 = """
SELECT 
    c.name,
    c.city,
    p.product_name,
    p.quantity
FROM read_csv_auto('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\customers.csv') c
JOIN read_json_auto('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\orders.json') j
ON c.customer_id = j.customer_id
JOIN read_parquet('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\products.parquet') p
ON j.order_id = p.order_id
WHERE c.city = 'Bangalore';
"""
df3 = con.execute(q3).fetchdf()
df3

,name,city,product_name,quantity
0,Neha Shah,Bangalore,Desk Lamp,1
1,Rohan Pillai,Bangalore,Desk Lamp,3
2,Rohan Pillai,Bangalore,Cotton Kurti,4
3,Neha Shah,Bangalore,Python Book,5
4,Neha Shah,Bangalore,Python Book,5
5,Sneha Mehta,Bangalore,Denim Jeans,2
6,Neha Shah,Bangalore,Denim Jeans,1
7,Aarav Sharma,Bangalore,Wireless Earbuds,5


In [16]:
q4 = """
SELECT 
    c.name,
    j.order_date,
    p.product_name,
    p.quantity
FROM read_csv_auto('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\customers.csv') c
JOIN read_json_auto('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\orders.json') j
ON c.customer_id = j.customer_id
JOIN read_parquet('E:\\massai\\codes\\python\\assignmet 2.0\\datasets\\products.parquet') p
ON j.order_id = p.order_id
ORDER BY j.order_date;
"""
df4 = con.execute(q4).fetchdf()
df4

,name,order_date,product_name,quantity
0,Rohan Gupta,2023-01-01,Face Wash,5
1,Arjun Singh,2023-01-07,Basmati Rice 5kg,4
2,Pooja Nair,2023-01-10,Wireless Earbuds,1
3,Amit Tiwari,2023-01-15,Yoga Mat,3
4,Rohan Mehta,2023-02-03,Wireless Earbuds,1
...,...,...,...,...
95,Kiran Chopra,2023-12-05,Running Shoes,4
96,Kiran Chopra,2023-12-05,Denim Jeans,4
97,Aarav Sharma,2023-12-17,Wireless Earbuds,5
98,Vikram Singh,2023-12-21,Yoga Mat,2


In [18]:
df1.to_csv("q1_output.csv", index=False)
df2.to_csv("q2_output.csv", index=False)
df3.to_csv("q3_output.csv", index=False)
df4.to_csv("q4_output.csv", index=False)